# 🤖 Lesson 01: What is an AI Agent?

Welcome to the AI Agents Course! In this first lesson, we explore what makes an AI agent different from a simple chatbot — and build our very first agent from scratch (no frameworks yet, just pure logic).

## 📦 Setup & Imports

In [ ]:
# Install if needed
# !pip install openai python-dotenv rich

import os
import json
from dotenv import load_dotenv
from openai import AzureOpenAI
from rich.console import Console
from rich.panel import Panel

load_dotenv()
console = Console()
console.print('[bold green]✅ Environment loaded![/bold green]')

## 🔌 Connect to Azure OpenAI

In [ ]:
client = AzureOpenAI(
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-02-15-preview')
)

DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o')
console.print(f'[bold blue]🔗 Connected to deployment: {DEPLOYMENT}[/bold blue]')

## 🤖 Part 1: Simple Chatbot (NOT an agent)

In [ ]:
def simple_chatbot(user_message: str) -> str:
    """A basic chatbot — no memory, no tools, no planning."""
    response = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_message}
        ],
        max_tokens=200
    )
    return response.choices[0].message.content

# Test it
reply = simple_chatbot('What is 2 + 2?')
console.print(Panel(reply, title='[yellow]Simple Chatbot Response[/yellow]'))

## 🧠 Part 2: Building the Agent Loop

Now let's build the **Perceive → Reason → Act** loop manually.

In [ ]:
# --- TOOLS (Actions the agent can take) ---
def calculate(expression: str) -> str:
    """Tool: Safely evaluate a math expression."""
    try:
        result = eval(expression, {'__builtins__': {}})
        return f'Result: {result}'
    except Exception as e:
        return f'Error: {str(e)}'

def get_current_topic(topic: str) -> str:
    """Tool: Return info about a DSA topic (simulated)."""
    topics = {
        'array': 'Arrays store elements in contiguous memory. Time: O(1) access, O(n) search.',
        'linked list': 'Linked Lists use nodes with pointers. Time: O(n) access, O(1) insert at head.',
        'binary search': 'Binary Search works on sorted arrays. Time: O(log n).',
        'stack': 'Stack is LIFO. Operations: push, pop, peek. All O(1).',
    }
    return topics.get(topic.lower(), f'Topic {topic} not found in database.')

# Tool registry
TOOLS = {
    'calculate': calculate,
    'get_current_topic': get_current_topic
}

console.print('[green]✅ Tools registered:[/green]', list(TOOLS.keys()))

In [ ]:
class SimpleAgent:
    """
    A minimal AI Agent with:
    - Perception: receives user input
    - Reasoning: uses LLM to decide action
    - Action: calls tools based on decision
    - Memory: keeps conversation history
    """
    
    def __init__(self, name: str):
        self.name = name
        self.memory = []  # Short-term memory
        self.system_prompt = """
You are a helpful AI assistant. You have access to these tools:
- calculate(expression): for math calculations
- get_current_topic(topic): to look up DSA concepts

When you need a tool, respond ONLY with valid JSON like:
{"action": "calculate", "input": "5 * 10"}

When you have a final answer, respond ONLY with:
{"action": "final_answer", "input": "your answer here"}
"""
    
    def perceive(self, user_input: str):
        """Step 1: Perceive the environment (receive input)."""
        self.memory.append({"role": "user", "content": user_input})
        console.print(f'[cyan]👁️  Perceiving:[/cyan] {user_input}')
    
    def reason(self) -> dict:
        """Step 2: Use LLM to reason and decide action."""
        messages = [{"role": "system", "content": self.system_prompt}] + self.memory
        
        response = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            max_tokens=300,
            temperature=0
        )
        
        raw = response.choices[0].message.content.strip()
        console.print(f'[yellow]🧠 Reasoning:[/yellow] {raw}')
        
        try:
            return json.loads(raw)
        except:
            return {"action": "final_answer", "input": raw}
    
    def act(self, decision: dict) -> str:
        """Step 3: Execute the decided action."""
        action = decision.get('action')
        input_val = decision.get('input', '')
        
        if action == 'final_answer':
            return input_val
        
        if action in TOOLS:
            result = TOOLS[action](input_val)
            console.print(f'[magenta]⚡ Action:[/magenta] {action}({input_val}) → {result}')
            # Add tool result to memory
            self.memory.append({"role": "assistant", "content": json.dumps(decision)})
            self.memory.append({"role": "user", "content": f'Tool result: {result}. Now give me the final answer.'})
            return None  # Signal to reason again
        
        return f'Unknown action: {action}'
    
    def run(self, user_input: str, max_steps: int = 5) -> str:
        """Run the full Perceive → Reason → Act loop."""
        console.print(Panel(f'[bold]Agent: {self.name}[/bold]\nTask: {user_input}', 
                           title='🚀 Agent Starting'))
        
        self.perceive(user_input)
        
        for step in range(max_steps):
            console.print(f'\n[bold]--- Step {step+1} ---[/bold]')
            decision = self.reason()
            result = self.act(decision)
            
            if result:  # Final answer reached
                console.print(Panel(result, title='[bold green]✅ Final Answer[/bold green]'))
                return result
        
        return 'Max steps reached.'


# --- Run the agent ---
agent = SimpleAgent(name='Avnish-Agent-v0')
agent.run('What is 125 multiplied by 8?')

In [ ]:
# Test with DSA topic lookup
agent2 = SimpleAgent(name='DSA-Helper')
agent2.run('Tell me about Binary Search and what its time complexity is.')

## 🔍 What did we just build?

```
User Input → [PERCEIVE] → stored in memory
                            ↓
                        [REASON] → LLM decides: tool or answer?
                            ↓
                         [ACT] → calls tool OR returns answer
                            ↓ (if tool used)
                        [REASON] → LLM sees tool result, gives final answer
```

This is the **core loop** every agent framework (AutoGen, LangChain, CrewAI) builds on!

## 🧪 Key Takeaways

1. An agent = LLM + Memory + Tools + Loop
2. A chatbot just responds; an agent **acts**
3. The Perceive→Reason→Act loop is universal
4. We'll use AutoGen from Lesson 03 to do all this automatically!